In [ ]:
from huggingface_hub import login
login(token="Enter your token here")

In [ ]:
!pip install auto-gptq
!pip install "transformers[sentencepiece]==4.32.1" "optimum==1.12.0" "auto-gptq==0.4.2" "accelerate==0.22.0" "safetensors>=0.3.1" --upgrade


In [ ]:
dataset = "wikitext2" 

In [ ]:
from optimum.gptq import GPTQQuantizer

quantizer = GPTQQuantizer(bits=4, dataset=dataset, model_seqlen=2048)
quantizer.quant_method = "gptq"

In [ ]:
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
from accelerate import infer_auto_device_map, load_checkpoint_and_dispatch

# Model ID and checkpoint file path
model_id = "meta-llama/Meta-Llama-3-8B"

# Directory path
directory = "C:\\Users\\kunal\\Desktop\\Projects\\ResTrack\\"

#Create the directory if it does not exist
if not os.path.exists(directory):
    os.makedirs(directory)

# Checkpoint file path
checkpoint_file = os.path.join(directory, "checkpoint.pt")

# Load the pre-trained model
model = AutoModelForCausalLM.from_pretrained(model_id,torch_dtype=torch.float16)

# Save the model's state dictionary 
torch.save(model.state_dict(), checkpoint_file)

# Load the state dictionary from the checkpoint
state_dict = torch.load(checkpoint_file)

# Load the state dictionary into the model
model.load_state_dict(state_dict)

# Initialize the model with empty weights for dispatching
from accelerate import init_empty_weights

with init_empty_weights():
    model = AutoModelForCausalLM.from_config(model.config)

# Infer the device map
device_map = infer_auto_device_map(model, max_memory="auto")

# Load the checkpoint and dispatch the model
model = load_checkpoint_and_dispatch(
    model, checkpoint=checkpoint_file, device_map=device_map, dtype=torch.float16
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False)


In [ ]:
quantized_model.push_to_hub("kunal1704/Meta-Llama-3-8B-Quantized")

In [ ]:
tokenizer.push_to_hub("kunal1704/Meta-Llama-3-8B-Quantized")